In [1]:
from Data_query.trino_config import *
import numpy as np
from visualisation import *
import pytz

In [26]:
stop_trino()

Trino service stopping triggered.


In [27]:
sleep(120)


In [29]:
sleep(180)

In [40]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(60)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [38]:
iceberg_exec("DROP TABLE IF EXISTS structured_data")
iceberg_exec("""CREATE TABLE structured_data (
                site_id BIGINT,
                t_stamp TIMESTAMP,
               actual_day DATE,
             actual_tod Time,
                  V DOUBLE,
             Q_kvar_norm DOUBLE,
             P_kw_norm DOUBLE,
             S_norm DOUBLE,
             GHI DOUBLE,
             cloud_type INT,
             cs_day DATE,
             cs_tod TIME,
             P_kw_norm_cs DOUBLE,
             GHI_cs DOUBLE,
             cloud_type_cs INT,
             S_99 DOUBLE,
                year INT,
                month INT
            )
             WITH (
                format = 'PARQUET',
                partitioning = ARRAY['year', 'month'],
                sorted_by = ARRAY['site_id', 't_stamp']
             )
    """)

Executed
Executed


In [36]:
sleep(20)

In [45]:
num_parts=2
time_bin_interval = '5' # in minutes
def run_func(args):
    year, month, split_cons, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and {part_filter}"
    df = iceberg_exec(f"""
                    insert into structured_data
                    with data as 
                        (select 
                            site_id, t_stamp,  sum(power*circuit_polarity)/1000/max(S_99) as P_kw_norm,
                            sum(energy_reactive*circuit_polarity)/1000/max(S_99)*12 as Q_kvar_norm, 
                            avg(voltage) as V,
                            max(S_99) as S_99
                        from ts join 
                            (select site_id, circuit_id, circuit_polarity, S_99  from meta_up23c where {meta_filters})
                            as m on ts.circuit_id = m.circuit_id
                        where {time_filter} and {part_filter} and is_pv=True and voltage > 0 and voltage < 300 and {split_cons}
                        group by site_id, t_stamp
                            ),
                    bom10min as (
                        select 
                            distinct time, b.latitude, b.longitude, surface_global_irradiance as GHI, cloud_type
                        from bom_nci.solar as b
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) as m 
                            on b.latitude = m.n_lat and b.longitude = m.n_long
                        where {time_filter} 
                    ),
                    bom5min as ((select time as time_5min, latitude, longitude, GHI, cloud_type
                    from bom10min) 
                    union all
                    (select date_add('minute', 5, time) as time_5min, latitude, longitude, GHI, cloud_type
                    FROM bom10min ORDER BY time_5min)),
                    daily_cloud AS (
                        SELECT
                            latitude, longitude,
                            date_trunc('day', time + interval '10' hour)   AS day,
                            date_trunc('month', time + interval '10' hour) AS month,
                            sum(cloud_type) AS cloud_sum, 
                            max(GHI) AS max_GHI
                        FROM bom10min
                        GROUP BY
                            1, 2, 3, 4
                    ),
                    clear_sky AS (
                            SELECT day, latitude, longitude
                            FROM (SELECT day, latitude, longitude, cloud_sum, max_GHI, row_number() OVER (
                                                                    PARTITION BY month, latitude, longitude
                                                                    ORDER BY cloud_sum ASC, day ASC
                                                                    ) AS rn
                                FROM daily_cloud 
                            )
                            WHERE rn < 4 and cloud_sum < 60 and max_GHI > 200
                        ),
                    daily_site_days AS (
                            SELECT 
                                n_lat,
                                n_long,
                                date_trunc('day', t_stamp + interval '10' hour) AS day
                            FROM data d join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            on d.site_id = m.site_id
                            group by n_lat, n_long, date_trunc('day', t_stamp + interval '10' hour) 
                        ),
                    nearest_clear_sky_day AS (
                        SELECT
                            dy.n_lat,
                            dy.n_long,
                            dy.day AS actual_day,
                            c.day AS clear_sky_day,
                            row_number() OVER (
                                PARTITION BY dy.n_lat, dy.n_long, dy.day
                                ORDER BY abs(date_diff('day', dy.day, c.day)), date_diff('day', c.day, dy.day)
                            ) AS rn
                        FROM daily_site_days dy JOIN clear_sky c
                            ON dy.n_lat = c.latitude AND dy.n_long = c.longitude
                    ),
                    nearest_clear_sky AS (
                            SELECT
                                n_lat,
                                n_long,
                                actual_day,
                                clear_sky_day
                            FROM nearest_clear_sky_day
                            WHERE rn = 1
                        ),
                    nearest_cs_days AS (
                        SELECT
                            DISTINCT site_id, clear_sky_day AS cs_day 
                        FROM nearest_clear_sky n JOIN (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            ON n.n_lat = m.n_lat AND n.n_long = m.n_long
                    ),
                    base AS (
                            SELECT
                                d.*,
                                lag(t_stamp) OVER (
                                    PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                    ORDER BY t_stamp
                                ) AS prev_ts
                            FROM data d
                    ),
                    gaps AS (
                        SELECT *,
                            CASE
                                WHEN prev_ts IS NULL THEN 0
                                WHEN t_stamp - prev_ts > interval '30' minute THEN 1
                                ELSE 0
                            END AS gap_start
                        FROM base
                    ),
                    segments AS (
                        SELECT *,
                            sum(gap_start) OVER (
                                PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                ORDER BY t_stamp
                                ROWS UNBOUNDED PRECEDING
                            ) AS segment_id
                        FROM gaps
                    ),
                    nearest_cs_profiles AS (
                        SELECT
                            s.site_id,
                            date_trunc('day', s.t_stamp + interval '10' hour) AS cs_day,
                            CAST(
                                date_trunc('minute', s.t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(s.t_stamp + interval '10' hour) % 5)
                                AS TIME) AS cs_tod,
                            approx_percentile(P_kw_norm, 0.6) OVER (
                                    PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                        ORDER BY t_stamp
                                        ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                                    ) AS P_kw_norm_cs,
                            approx_percentile(GHI, 0.6) OVER (
                            PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                ORDER BY t_stamp
                                ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                            ) AS GHI_cs,
                            cloud_type as cloud_type_cs
                        FROM segments s join nearest_cs_days n on s.site_id = n.site_id and 
                            date_trunc('day', s.t_stamp + interval '10' hour) = n.cs_day
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m on s.site_id = m.site_id
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = s.t_stamp
                    ),
                    strcutured_data AS (
                        SELECT
                            d.site_id,
                            d.t_stamp,
                            date_trunc('day', d.t_stamp + interval '10' hour) AS actual_day,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % 5)
                                AS TIME) AS actual_tod,
                            d.V,
                            d.Q_kvar_norm,
                            d.P_kw_norm, 
                            sqrt(pow(d.Q_kvar_norm, 2) + pow(d.P_kw_norm, 2)) AS S_norm,
                            GHI,
                            cloud_type,
                            ncs.cs_day,
                            ncs.cs_tod,
                            ncs.P_kw_norm_cs,
                            ncs.GHI_cs, 
                            ncs.cloud_type_cs,
                            d.S_99,
                            year(t_stamp) as year, 
                            month(t_stamp) as month
                        FROM data d 
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                                ON d.site_id = m.site_id 
                            join nearest_cs_profiles ncs 
                                on d.site_id = ncs.site_id and ncs.cs_tod = CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % 5)
                                AS TIME)
                            join nearest_clear_sky n on 
                                n.n_lat = m.n_lat AND n.n_long = m.n_long and n.actual_day = date_trunc('day', d.t_stamp + interval '10' hour)
                                and n.clear_sky_day = ncs.cs_day
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = d.t_stamp
                        Where abs(date_diff('day', ncs.cs_day, n.actual_day)) < 45
                            order by d.site_id, d.t_stamp)
                        select * from strcutured_data
                        """)

                    # 

    # sleep(10)
    print(f"Completed {time_filter}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}, part {part}")
    return df
tasks = [(year, month, split_cons, part) for year in (2024, ) for month in range(1, 2) 
         for split_cons in [f'system.bucket(postcode, 16) = {i}' for i in range(14, 16)]
         for part in range(0, num_parts)]
        #   for split_cons in ['system.bucket(postcode, 16) > -1'] ]
            
try:         
    df = trino_parallel_batch(run_func, tasks, num_workers=num_workers, batch_size=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
# df['t_stamp'] = pd.to_datetime(df['t_stamp']).dt.tz_localize('utc').dt.tz_convert(pytz.FixedOffset(10*60))
df

Executed
Completed year = 2024, bucket = 14, part 0
Executed
Completed year = 2024, bucket = 14, part 1
Executed
Completed year = 2024, bucket = 15, part 0
Executed
Completed year = 2024, bucket = 15, part 1


In [3]:
sleep(40)

In [46]:
iceberg_sql(f"""select count(distinct site_id) from structured_data
            """)

,_col0
0,13967
